In [1]:
import torch
inputs = torch.tensor(
    [[
        [0.43, 0.15, 0.89], # Your     
        [0.55, 0.87, 0.66], # journey  
        [0.57, 0.85, 0.64], # starts   
        [0.22, 0.58, 0.33], # with     
        [0.77, 0.25, 0.10], # one      
        [0.05, 0.80, 0.55]  # step     
    ],
    [
        [0.43, 0.15, 0.89], # His     
        [0.55, 0.87, 0.66], # journey  
        [0.57, 0.85, 0.64], # end   
        [0.22, 0.58, 0.33], # at     
        [0.77, 0.25, 0.10], # last      
        [0.05, 0.80, 0.55]  # way     
    ]
])

In [2]:
inputs.shape   # no_of_batch, no_of_token, embedding_dim

torch.Size([2, 6, 3])

In [5]:
import torch.nn as nn

class MaskedSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias = False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.d_out = d_out
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer(
            "mask",
            torch.triu( torch.ones(context_length,context_length) , diagonal=1 )
        )
        
    def forward(self,x):
        no_of_batch, no_of_token, embedding_dim = x.shape
        
        
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        # attention = Q @ (K)T
        attention_score = queries @ keys.transpose(1,2)
        
        # masking
        attention_score = attention_score.masked_fill(
            self.mask.bool()[:no_of_token, :no_of_token], -torch.inf
        )
        
        # normalize
        d_k = keys.shape[-1]
        attention_weight = torch.softmax(
            attention_score/d_k**0.5, dim=-1
        )
        
        # dropout
        attention_weight = self.dropout(attention_weight)
        
        context_vector = attention_weight @ values
        return context_vector

In [6]:
d_in = inputs.shape[-1]         
d_out = 2
context_length = inputs.shape[1] 
print(d_in, d_out, context_length)

3 2 6


In [7]:
maskedSelfAtten = MaskedSelfAttention(d_in, d_out, context_length, dropout=0.5)

In [8]:
context_length = maskedSelfAtten(inputs)
context_length

tensor([[[ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [-0.3249, -0.5582],
         [-0.4712, -0.4464],
         [-0.6345, -0.6649],
         [-0.3828, -0.4510]],

        [[ 0.0000,  0.0000],
         [-0.4815, -0.8269],
         [-0.6378, -0.8519],
         [-0.6351, -0.7675],
         [-0.0069, -0.1499],
         [-0.4236, -0.5113]]], grad_fn=<UnsafeViewBackward0>)